# GEM TREQ: exposición de edificios recortada a siete comunas

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad

Una fila representa un edificio del modelo de exposición GEM, recortado a las siete comunas del inventario. `night` y vulnerabilidad son estimaciones del modelo; no observaciones de ocupación o daño durante el sismo.

Este archivo fue materializado por `scripts/descargar_externos.py`. Se distingue de los originales remotos: el script recorta, combina o deriva columnas antes de escribirlo.

In [ ]:
PATH=ROOT/'data/externos/gem_exposicion.csv'
meta=ficha(PATH)
df=pd.read_csv(PATH,keep_default_na=False)
calidad=mostrar_perfil(df)
x=df.copy()
g=gpd.GeoDataFrame(x,geometry=gpd.points_from_xy(pd.to_numeric(x.lon),pd.to_numeric(x.lat)),crs=4326)
geo=geometria(g)
display(pd.Series({**meta, "filas":len(df),"columnas":len(df.columns)}).to_frame("valor"))
display(pd.Series(geo).to_frame("geometria"))
resumen={"dataset":'13_gem_exposicion',**meta,"filas":len(df),"columnas":list(df),"calidad_columnas":calidad.to_dict("records"),"geometria":geo}

## 2. Llaves, categorías, probabilidades y discretización de coordenadas

In [ ]:
for c in ["CONSTRUCTION_PERIOD","INCOME_LEVEL","FLOORS","OCCUPANCY","COMUNA"]:display(conteos(df[c],c))
probs=df[["p_ductilidad_baja","p_mamposteria_no_reforzada"]].apply(pd.to_numeric,errors="coerce")
coord_4 = np.isclose(df.lon*10000,np.round(df.lon*10000),rtol=0,atol=1e-6)&np.isclose(df.lat*10000,np.round(df.lat*10000),rtol=0,atol=1e-6)
qa={"id_nulos":int(df.id.isna().sum()),"id_duplicados":int(df.id.duplicated().sum()),
    "puntos_coincidentes_adicionales":int(df.duplicated(["lon","lat"]).sum()),
    "filas_en_puntos_repetidos":int(df.duplicated(["lon","lat"],keep=False).sum()),
    "coordenadas_compatibles_4_decimales":int(coord_4.sum()),
    "probabilidades_fuera_0_1":int(((probs<0)|(probs>1)).sum().sum()),
    "probabilidades_nulas":int(probs.isna().sum().sum()),
    "ocupantes_noche_negativos":int(df.night.lt(0).sum()),"valor_reemplazo_negativo":int(df.structural.lt(0).sum())}
display(pd.Series(qa).to_frame("control"))
display(df[["night","structural",*probs.columns]].describe(percentiles=[.5,.95,.99]))
grafico_barras(conteos(df.FLOORS,"rango_pisos"),"Altura clasificada en el modelo GEM", "13_altura")
resumen.update(qa)
resumen["rangos_pisos"]=df.FLOORS.value_counts().to_dict()
resumen["comunas"]=df.COMUNA.value_counts().to_dict()

## Reglas decididas para plata

- **Seleccionar como enriquecimiento condicionado a calidad del emparejamiento.** El ID GEM no es OBJECTID del evento.
- El extractor redondea todo el CSV a cuatro decimales, incluidas coordenadas. Conservar los originales remotos sin redondeo antes de asumir precisión predial o emparejamiento exacto. Auditar ambigüedades y reutilización de puntos GEM en la síntesis.
- Preservar `FLOORS` como categoría. `6-10` y `11+` no son alturas exactas; imputarlas como 8 o 12 exige bandera, rango e hipótesis explícita.
- `night` es ocupación nocturna modelada; las probabilidades proceden de una conversión de tipologías con pesos expertos. Guardar fuente y versión de esa conversión; no interpretarlas como daño observado o conteo censal actual.
- No trasladar atributos entre edificios por tomar simplemente la primera coincidencia espacial. Separar coincidencias únicas, ambiguas y sin soporte.
- No usar la coincidencia GEM para convertir un lote oficial (`NPISOS=0`) en un edificio. Para pisos NULL, cualquier estimación requiere correspondencia validada y bandera de origen.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('13_gem_exposicion', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")